In [1]:
import uuid
from datetime import datetime

PIPELINE_NAME = "04_5_gold_business_metrics"
LAYER = "gold"
TABLE_NAME = "business_metrics_gold"
LOG_TABLE = "pipeline_run_log"

run_id = str(uuid.uuid4())
start_time = datetime.now()

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {LOG_TABLE} (
    pipeline_run_id STRING,
    pipeline_name STRING,
    layer STRING,
    table_name STRING,
    status STRING,
    start_time TIMESTAMP,
    end_time TIMESTAMP,
    input_rows BIGINT,
    output_rows BIGINT,
    dq_status STRING
)
USING DELTA
""")

def log_start():
    spark.sql(f"""
    INSERT INTO  {LOG_TABLE}
    VALUES (
        '{run_id}',
        '{PIPELINE_NAME}',
        '{LAYER}',
        '{TABLE_NAME}',
        'RUNNING',
        CURRENT_TIMESTAMP(),
        NULL,
        NULL,
        NULL,
        NULL

    )
    """)

def log_finish(status, input_rows=None, output_rows=None,dq_status=None):

    input_rows_sql = "NULL" if input_rows is None else str(input_rows)
    output_rows_sql = "NULL" if output_rows is None else str(output_rows)
    dq_status_sql = "NULL" if dq_status is None else str(dq_status)

    spark.sql(f"""
    MERGE INTO {LOG_TABLE}  t
    USING (
        SELECT '{PIPELINE_NAME}' AS pipeline_name,
        '{run_id}' AS pipeline_run_id
    ) s
    ON t.pipeline_name = s.pipeline_name
    AND t.pipeline_run_id = s.pipeline_run_id
    WHEN MATCHED THEN UPDATE SET
        t.status = '{status}',
        t.end_time = CURRENT_TIMESTAMP(),
        t.input_rows = {input_rows_sql},
        t.output_rows = {output_rows_sql},
        t.dq_status = '{dq_status_sql}'
    """)

StatementMeta(, fc05922f-9ec9-4359-9908-7367050e6975, 3, Finished, Available, Finished, False)

In [2]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

REVIEW = "fact_review"
BUSINESS = "dim_business"
GOLD = "business_metrics_gold" 

df_review = spark.table(REVIEW)
df_business = spark.table(BUSINESS)

df_metric = (
    df_review.alias("r")
    .join(df_business.alias("b"), on="business_id", how="left")
    .groupBy("business_id", "name", "city", "state", "categories")
    .agg(
        F.count("r.review_id").alias("review_count_total"),
        F.round(F.avg("r.stars"),2).alias("avg_rating"),
        F.min("r.date_id").alias("first_review_date_id"),
        F.max("r.date_id").alias("last_review_date_id")
    )
    .withColumn(
        "business_popularity_tier",
        F.when(F.col("review_count_total") >= 500, "high")
        .when(F.col("review_count_total") >= 100, "medium")
        .otherwise("low")
    )
    .withColumn(
        "rating_bucket",
        F.when(F.col("avg_rating") >= 4.5, "excellent")
        .when(F.col("avg_rating") >= 3.5, "good")
        .when(F.col("avg_rating") >= 2.5, "average")
        .otherwise("poor")
    )
)


StatementMeta(, fc05922f-9ec9-4359-9908-7367050e6975, 4, Finished, Available, Finished, False)

In [3]:
input_rows = df_review.count()

StatementMeta(, fc05922f-9ec9-4359-9908-7367050e6975, 5, Finished, Available, Finished, False)

In [4]:
# ======================== DQ start ========================== #

StatementMeta(, fc05922f-9ec9-4359-9908-7367050e6975, 6, Finished, Available, Finished, False)

In [5]:
output_rows = df_metric.count()

StatementMeta(, fc05922f-9ec9-4359-9908-7367050e6975, 7, Finished, Available, Finished, False)

In [6]:
def run_gold_business_metrics_dq(df):

    # Gold DQ for business_metrics_gold
    # Returns:
    #     dq_df: rule-level DQ result dataframe
    # Raises:
    #     Exception if any rule fails
    
    total_rows = df.count()
    results = []

    def add_result(
        rule_name, 
        dimension, 
        column_name, 
        failed_count,
        threshold,
        notes=""
        ):
        status = "PASS" if failed_count <= threshold else "FAIL"
        results.append({
            "rule_name": rule_name, 
            "dimension": dimension, 
            "column_name": column_name, 
            "failed_count": int(failed_count),
            "threshold": int(threshold),
            "status": status,
            "notes": notes
        })

    # 1. Table checks 
    add_result(
            rule_name = "row_count_gt_0", 
            dimension = "completeness", 
            column_name = "*", 
            failed_count = 0 if total_rows > 0 else 1,
            threshold = 0,
            notes = f"total_rows = {total_rows}"
    )

    # 2. Key checks
    null_business_id = df.filter(F.col("business_id").isNull()).count()
    add_result(
        rule_name = "business_id_not_null", 
        dimension = "completeness", 
        column_name = "business_id", 
        failed_count = null_business_id,
        threshold = 0
    )

    dup_business_id = (
        df.groupBy("business_id")
        .count()
        .filter(F.col("business_id").isNotNull() & (F.col("count") > 1))
        .count()
    )
    add_result(
        rule_name="business_id_unique",
        dimension="uniqueness",
        column_name="business_id",
        failed_count=dup_business_id,
        threshold = 0,
        notes="count of duplicated business_id groups"       
    )

    join_unmatched_rows = (
        df_review.alias("r")
        .join(df_business.alias("b"), on="business_id", how="left")
        .filter(F.col("r.business_id").isNotNull() & F.col("b.business_id").isNull())
        .count()
    )
    add_result(
        rule_name="business_id_join_match",
        dimension="referential_integrity",
        column_name="business_id",
        failed_count=join_unmatched_rows,
        threshold = 0,
        notes = "review.business_id should exist in dim_business.business_id"
    )

    # 3. Descriptive dimension checks
    null_name = df.filter(F.col("name").isNull()).count()
    add_result(
        rule_name = "name_not_null", 
        dimension = "completeness", 
        column_name = "name", 
        failed_count = null_name,
        threshold = 0
    )

    null_city = df.filter(F.col("city").isNull()).count()
    add_result(
        rule_name = "city_not_null", 
        dimension = "completeness", 
        column_name = "city", 
        failed_count = null_city,
        threshold = 0,
    )

    null_state = df.filter(F.col("state").isNull()).count()
    add_result(
        rule_name = "state_not_null", 
        dimension = "completeness", 
        column_name = "state", 
        failed_count = null_state,
        threshold = 0,
    )

    null_categories = df.filter(F.col("categories").isNull()).count()
    null_ratio = null_categories / total_rows
    add_result(
        rule_name = "categories_null_ratio", 
        dimension = "completeness", 
        column_name = "categories", 
        failed_count = null_categories,
        threshold = int(total_rows * 0.05),
        notes = "allow up to 5% null categories./threshold=0 means category is not allowed to be null"
    )

    #4. Metric validity checks
    bad_review_count = df.filter(F.col("review_count_total") < 0).count()
    add_result(
        rule_name = "review_count_total_gte_0", 
        dimension = "validity", 
        column_name = "review_count_total", 
        failed_count = bad_review_count,
        threshold = 0,
        notes = ""
    )

    null_review_count = df.filter(F.col("review_count_total").isNull()).count()
    add_result(
        rule_name = "review_count_total_not_null", 
        dimension = "completeness", 
        column_name = "review_count_total", 
        failed_count = null_review_count,
        threshold = 0,
    )

    bad_avg_rating = df.filter(
        (F.col("avg_rating").isNull()) |
        (F.col("avg_rating") < 1.0)|
        (F.col("avg_rating") > 5.0)
    ).count()
    add_result(
        rule_name = "avg_rating_between_1_and_5", 
        dimension = "validity", 
        column_name = "avg_rating", 
        failed_count = bad_avg_rating,
        threshold = 0
    )

    ##review_count_total > 0, avg_rating should not be null
    bad_avg_rating_null_logic = df.filter(
        (F.col("review_count_total") > 0) &
        F.col("avg_rating").isNull()
    ).count()
    add_result(
        rule_name = "avg_rating_not_null_when_reviews_exist", 
        dimension = "consistency", 
        column_name = "avg_rating", 
        failed_count = bad_avg_rating_null_logic,
        threshold = 0
    )

    #5. Date consistency checks
    null_first_date = df.filter(F.col("first_review_date_id").isNull()).count()
    add_result(
        rule_name = "first_review_date_id_not_null", 
        dimension = "completeness", 
        column_name = "first_review_date_id", 
        failed_count = null_first_date,
        threshold = 0
    )

    null_last_date = df.filter(F.col("last_review_date_id").isNull()).count()
    add_result(
        rule_name = "last_review_date_id_not_null", 
        dimension = "completeness", 
        column_name = "last_review_date_id", 
        failed_count = null_last_date,
        threshold = 0
    )

    bad_date_order = df.filter(
        F.col("first_review_date_id").isNotNull() &
        F.col("last_review_date_id").isNotNull() &
        (F.col("first_review_date_id") > F.col("last_review_date_id"))
    ).count()
    add_result(
        rule_name = "first_review_date_id_lte_last_review_date_id", 
        dimension = "consistency", 
        column_name = "first_review_date_id, last_review_date_id", 
        failed_count = bad_date_order,
        threshold = 0
    )

    #6. Derived-column checks
    bad_tier = df.filter(
        ~F.col("business_popularity_tier").isin("high", "medium", "low")
    ).count()
    add_result(
        rule_name="business_popularity_tier_valid_values",
        dimension="validity",
        column_name="business_popularity_tier",
        failed_count=bad_tier,
        threshold=0
    )

    bad_tier_logic = df.filter(
        ((F.col("review_count_total") >= 500) & (F.col("business_popularity_tier") != "high")) |
        ((F.col("review_count_total") >= 100) & (F.col("review_count_total") < 500) & (F.col("business_popularity_tier") != "medium")) |
        ((F.col("review_count_total") < 100) & (F.col("business_popularity_tier") != "low"))
    ).count()
    add_result(
        rule_name="business_popularity_tier_logic_match",
        dimension="consistency",
        column_name="business_popularity_tier",
        failed_count=bad_tier_logic,
        threshold=0
    )

    bad_bucket = df.filter(
        ~F.col("rating_bucket").isin("excellent", "good", "average", "poor")
    ).count()
    add_result(
        rule_name = "rating_bucket_valid_values", 
        dimension = "validity", 
        column_name = "rating_bucket", 
        failed_count = bad_bucket,
        threshold =0
    )

    ##tier logical consistency
    bad_bucket_logic = df.filter(
        ((F.col("avg_rating") >= 4.5) & (F.col("rating_bucket") != "excellent")) |
        ((F.col("avg_rating") >= 3.5) & (F.col("avg_rating") < 4.5) & (F.col("rating_bucket") != "good")) |
        ((F.col("avg_rating") >= 2.5) & (F.col("avg_rating") < 3.5) & (F.col("rating_bucket") != "average")) |
        ((F.col("avg_rating") < 2.5) & (F.col("rating_bucket") != "poor"))
    ).count()
    add_result(
        rule_name = "rating_bucket_logic_match", 
        dimension = "consistency", 
        column_name = "rating_bucket",
        failed_count = bad_bucket_logic,
        threshold = 0
    )

    #7. Join-quality / orphan check
    core_dim_missing_rows = df.filter(
        F.col("business_id").isNotNull() &
        F.col("name").isNull() &
        F.col("city").isNull() &
        F.col("state").isNull()
    ).count()
    add_result(
        rule_name = "business_dim_core_fields_not_all_null", 
        dimension = "consistency", 
        column_name = "name,city,state",
        failed_count = core_dim_missing_rows,
        threshold =0,
        notes = "business matched successfully, but name/city/state are all null"
    )

    dq_df = spark.createDataFrame(results)
    
    fail_count = dq_df.filter(F.col("status") == "FAIL").count()

    display(dq_df.orderBy("status", "rule_name"))

    if fail_count > 0:
        failed_rules = (
            dq_df.filter(F.col("status") == "FAIL")
            .select("rule_name", "failed_count", "notes")
            .collect()
        )

        error_lines = ["Gold DQ failed for business_metrics_gold:"]
        for r in failed_rules:
            error_lines.append(
                f"- {r['rule_name']}: failed_count = {r['failed_count']}; notes = {r['notes']}"
            )

        raise Exception("\n".join(error_lines))

    print("Gold DQ passed for business_metrics_gold.")
    return dq_df


StatementMeta(, fc05922f-9ec9-4359-9908-7367050e6975, 8, Finished, Available, Finished, False)

In [7]:
dq_result = run_gold_business_metrics_dq(df_metric)

StatementMeta(, fc05922f-9ec9-4359-9908-7367050e6975, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5192442e-2b4c-4cc8-adfb-b4808f224516)

Gold DQ passed for business_metrics_gold.


In [8]:
dq_status = "PASSED"

StatementMeta(, fc05922f-9ec9-4359-9908-7367050e6975, 10, Finished, Available, Finished, False)

In [9]:
# ======================== DQ end ========================== #

StatementMeta(, fc05922f-9ec9-4359-9908-7367050e6975, 11, Finished, Available, Finished, False)

In [10]:
try:

    log_start()

    if spark.catalog.tableExists(GOLD):
        delta = DeltaTable.forName(spark, GOLD)

        (
            delta.alias("t")
            .merge(
                df_metric.alias("d"),
                "t.business_id = d.business_id"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
    else:
        (
            df_metric.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(GOLD)
        )

    log_finish(
        status = "SUCCESS",
        input_rows = input_rows,
        output_rows = output_rows,
        dq_status = dq_status
    )
except Exception as e:
    log_finish(
        status = "FAILED",
        input_rows = input_rows,
        output_rows = output_rows,
        dq_status = "FAILED"
    )
    raise


StatementMeta(, fc05922f-9ec9-4359-9908-7367050e6975, 12, Finished, Available, Finished, False)

In [11]:
display(spark.table("pipeline_run_log"))

StatementMeta(, fc05922f-9ec9-4359-9908-7367050e6975, 13, Finished, Available, Finished, True)

SynapseWidget(Synapse.DataFrame, 9cbdd1f0-f2c4-4c4a-a9f8-f5dc25a55870)